# Lab 7 - PySpark i SQL, wiaderkowanie i partycjonowanie plików oraz zapis w hurtowni danych Hive.

In [3]:
# dla instalacji lokalnej

import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_HOME'] = sys.executable

In [4]:
from pyspark.sql import SparkSession

# ścieżka do bazy danych hurtowni danych oraz plików
# należy dostosować do ścieżki względnej, w której umieszczony został bieżący notebook
warehouse_location = '../data/metastore'

# utworzenie sesji Spark, ze wskazaniem włączenia obsługi Hive oraz
# lokalizacją przechowywania hurtowni danych
spark = SparkSession\
        .builder\
        .master("local[2]")\
        .appName("Apache SQL and Hive")\
        .config("spark.memory.offHeap.enabled","true")\
        .config("spark.memory.offHeap.size","4g")\
        .config("spark.executor.memory", "2g") \
        .config("spark.driver.memory", "1g") \
        .config("spark.driver.host", "localhost")\
        .enableHiveSupport()\
        .config("spark.sql.warehouse.dir", warehouse_location)\
        .getOrCreate()

In [5]:
spark.sparkContext

<SparkContext master=local[2] appName=Apache SQL and Hive>

## 1. Spark i SQL

Spark umożliwia zarejestrowanie obiektu DataFrame jako widoku, co umożliwia korzystanie z niego w sposób bardzo zbliżony do pracy z językiem SQL. Poniżej przykład.

In [4]:
# dostosuj ścieżkę do pliku do swoich danych, tutaj został utworzony mniejszy zbiór niż w poprzednim labie
df = spark.read.csv('../data/employee.csv', header=True, inferSchema=True)

In [5]:
# tworzymy widok tymczasowy w pamięci węzła
df.createOrReplaceTempView("EMPLOYEE_DATA")

In [6]:
# wypisanie tabeli, zwróć uwagę na to, czy stworzona tabela jest tymczasowa czy trwała
spark.catalog.listTables()

[Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [8]:
# pobranie danych jak z tabeli SQL
# %%time
spark.sql("Select * from EMPLOYEE_DATA limit 4").show()
# spark.sql("select firstname from EMPLOYEE_DATA").show(10)

+---+---------+-------------------+---+-------+
| id|firstname|           lastname|age| salary|
+---+---------+-------------------+---+-------+
|  1| Zbigniew|Brzęczyszczykiewicz| 38| 6273.5|
|  2|    Marek|             Szczaw| 38|9583.23|
|  3|    Marek|       Mieczykowski| 53|5138.74|
|  4|     Adam|             Szczaw| 31|7930.89|
+---+---------+-------------------+---+-------+



In [9]:
spark.sql("select firstname, count(firstname), avg(salary) from EMPLOYEE_DATA group by firstname").show()

+----------+----------------+------------------+
| firstname|count(firstname)|       avg(salary)|
+----------+----------------+------------------+
|   Wisława|         1999590| 7849.980336614015|
|Mieczysław|         2002113|  7849.60820492158|
|     Agata|         1998582| 7849.972157664713|
| Krzysztof|         1998078|7849.8804712578485|
|     Marek|         1998080| 7850.454721973166|
|      Adam|         2000530| 7849.955449350843|
| Katarzyna|         2001396| 7851.613844691453|
|  Wojciech|         1999717| 7850.247844250032|
|  Zbigniew|         2001686|7849.5790366222045|
|Aleksandra|         2000228| 7849.161836555669|
+----------+----------------+------------------+



In [11]:
the_raise = 0.1 # 10% podwyżki
spark.sql(f"select firstname, lastname, salary, round(salary + salary * {the_raise},2) as with_raise from EMPLOYEE_DATA").show(5)

+---------+-------------------+--------+----------+
|firstname|           lastname|  salary|with_raise|
+---------+-------------------+--------+----------+
| Zbigniew|           Barański| 8797.82|    9677.6|
|Krzysztof|           Kowalski| 7441.71|   8185.88|
|Krzysztof|Brzęczyszczykiewicz| 8502.79|   9353.07|
|     Adam|         Wróblewski|10258.55|   11284.4|
|  Wisława|           Barański|  9006.9|   9907.59|
+---------+-------------------+--------+----------+
only showing top 5 rows



## 2. Apache Hive

https://hive.apache.org/


Apache Hive, który pierwotnie został stworzony w 2007 przez Facebooka, a następnie w 2008 przekazany pod skrzydła Apache Foundation, jest nazywany hurtownią danych. Dane przechowywane są głównie w systemie **HDFS** (**Hadoop Distributed File System**), ale Hive integruje się również z innymi silnikami baz danych.

Dostęp do danych jest realizowany przez **Hive QL**, który bardzo przypomina język SQL i taki sposób obsługi różnorodnych danych był jedną z głównych motywacji powstania Hive.

Za pomocą zapytań Hive QL (HQL) można wykonać takie zapytania jak:
* tworzenie i zmiana struktur tabel,
* import i export danych,
* agregacja danych, filtrowanie i złączenia danych.

Apache Hive jest wykorzystywany w dużych ekosystemach i mimo wymienionych wyżej zalet posiada również kilka ograniczeń:
* opóźnienie w czasie przetwarzania ze zwględu na wsadową naturę przetwarzania,
* brak możliwości przetwarzania real-time,
* język HQL nie daje możliwości wykonania takich operacji jak modyfikacja danych na poziomie wiersza,
* brak możliwości przeprowadzenia zaawansowanych analiz jak współczesne nowoczesne bazy SQL.

Alternatywne technologie:

* Presto
* Snowflake
* Apache Impala
* IBM Db2
* Google BigQuery
* Amazon Redshift
* ClickHouse
* Apache Hadoop
* Apache HBase
* Oracle Exadata
* Teradata Vantage
* Cloudera Impala

### 2.1 Hive QL

> Dokumentacja Apache Hive QL (dość archaiczna) jest dostępna pod adresem: https://cwiki.apache.org/confluence/display/Hive/LanguageManual

In [10]:
spark.catalog.currentCatalog()

'spark_catalog'

In [11]:
# dla zrealizowania kolejnych przykładów dokonamy kilku modyfikacji pliku employee
# 1. dodanie kolumny ID - indeksu
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn("ID", monotonically_increasing_id())

In [12]:
df.show(10)

+---+---------+-------------------+---+-------+
| ID|firstname|           lastname|age| salary|
+---+---------+-------------------+---+-------+
|  0| Zbigniew|Brzęczyszczykiewicz| 38| 6273.5|
|  1|    Marek|             Szczaw| 38|9583.23|
|  2|    Marek|       Mieczykowski| 53|5138.74|
|  3|     Adam|             Szczaw| 31|7930.89|
|  4|    Agata|           Kowalski| 57| 8688.3|
|  5| Zbigniew|           Kowalski| 26|5481.89|
|  6|     Adam|           Barański| 65|8272.78|
|  7|    Marek|           Kowalski| 18|6214.96|
|  8|     Adam|              Pysla| 55|6176.28|
|  9|Krzysztof|             Wlotka| 41|8396.87|
+---+---------+-------------------+---+-------+
only showing top 10 rows


In [13]:
# dokonamy podziału danych i zapisania w różnych formatach
splits = df.randomSplit(weights=[0.3, 0.7], seed=19)

In [14]:
splits[0].count(), splits[1].count()

(5998734, 14001266)

In [ ]:
# to dość dziwne zjawisko niezbyt równego podziału danych jest opisane w artykułach:
# https://medium.com/udemy-engineering/pyspark-under-the-hood-randomsplit-and-sample-inconsistencies-examined-7c6ec62644bc
# oraz
# https://www.geeksforgeeks.org/pyspark-randomsplit-and-sample-methods/

In [15]:
# większa część trafi do nowej tymczasowej tabeli
splits[1].createOrReplaceTempView("EMPLOYEE_DATA_SPLIT_1")

In [16]:
spark.catalog.listTables()

[Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employee_data_split_1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [17]:
# a mniejsza do plików JSON
splits[0].write.json('../data/json/employee', mode='overwrite')

In [18]:
!ls ../data/json/employee/*.json

../data/json/employee/part-00000-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json
../data/json/employee/part-00001-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json
../data/json/employee/part-00002-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json
../data/json/employee/part-00003-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json
../data/json/employee/part-00004-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json
../data/json/employee/part-00005-a98d1f89-dba7-415d-8dac-99f4d2097c2d-c000.json


In [19]:
# aby móc wykorzystać dane w przykładach ze złączaniem, zapiszemy jeszcze próbkę danych z głównej ramki
# z identyfikatorami oraz dodatkową kolumną z podwyżką
from pyspark.sql.functions import col, lit, round

lucky_guys = spark.sql("select * from EMPLOYEE_DATA").sample(0.01)\
.withColumn('raise', lit('10%')).withColumn('salary after raise', round(col('salary') * 1.1, 2))

In [22]:
lucky_guys

DataFrame[id: int, firstname: string, lastname: string, age: int, salary: double, raise: string, salary after raise: double]

In [23]:
# zapisujemy szczęściarzy do oddzielnej tabeli w hurtowni
lucky_guys.write.mode('overwrite').saveAsTable("lucky_employees", format='parquet')

In [24]:
spark.catalog.listTables()

[Table(name='lucky_employees', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employee_data_split_1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

#### Złączenie danych z różnych źródeł

In [25]:
!ls ../data/metastore/lucky_employees/*.parquet

../data/metastore/lucky_employees/part-00000-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet
../data/metastore/lucky_employees/part-00001-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet
../data/metastore/lucky_employees/part-00002-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet
../data/metastore/lucky_employees/part-00003-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet
../data/metastore/lucky_employees/part-00004-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet
../data/metastore/lucky_employees/part-00005-066ee8ff-f082-46d0-81ba-b12d1a4834ef-c000.snappy.parquet


In [26]:
# przykład złączania danych na różnych źródłach
# zapytanie SQL bezpośrednio na plikach - tutaj zapisanych wcześniej JSON-ach oraz parquet
query = """
SELECT ed.ID, ed.firstname, ed.lastname, ed.salary, lucky.raise, lucky.`salary after raise`
FROM json.`../data/json/employee/` as jtable 
join EMPLOYEE_DATA ed on jtable.ID=ed.ID 
join parquet.`../data/metastore/lucky_employees/` as lucky on ed.ID=lucky.ID
"""
df_from_json = spark.sql(query).show(10)

+-----+----------+-------------------+-------+-----+------------------+
|   ID| firstname|           lastname| salary|raise|salary after raise|
+-----+----------+-------------------+-------+-----+------------------+
|10514|      Adam|         Malinowski|7206.04|  10%|           7926.64|
|10630|Mieczysław|Brzęczyszczykiewicz|9169.37|  10%|          10086.31|
|20064|Aleksandra|         Malinowski| 6931.3|  10%|           7624.43|
|21065| Krzysztof|             Wlotka|9602.76|  10%|          10563.04|
|22784|  Zbigniew|             Szczaw|6800.65|  10%|           7480.72|
|23406|     Agata|           Barański|8955.83|  10%|           9851.41|
|24612|Mieczysław|         Wróblewski|8560.75|  10%|           9416.83|
|36294| Katarzyna|             Szczaw|6312.32|  10%|           6943.55|
|38245|Aleksandra|       Mieczykowski|7100.37|  10%|           7810.41|
|40415|Aleksandra|         Wróblewski|9071.81|  10%|           9978.99|
+-----+----------+-------------------+-------+-----+------------

#### Dzielenie danych na wiaderka (ang. buckets) i partycje

Dzielenie danych na wiaderka jest rozwiązaniem, które stosowane jest do podziału danych na mniejsze części w sposób, który może przyspieszyć obliczenia poprzez zredukowanie liczby operacji przetasowania danych (ang. shuffle, a w kontekście Sparka mówimy o operacji exchange), które są bardzo kosztowne, gdyż wykonywane są często między węzłami (workerami).

In [27]:
# ten przykład pokazuje podział na 16 wiaderek danych bazując na podziale po kolumnie ID (tu używane jest hashowanie)
# dane posortowane są w każdym buckecie po kolumnie salary
# dane zapisywane są do hurtowni Hive, a informacje o zapisanych tam tabelach przechowywane są w
# Hive metastore (domyślnie jest do baza danych Derby)
df.write.bucketBy(16, 'ID').mode('overwrite').sortBy('salary').saveAsTable('employee_id_bucketed')

In [ ]:
!ls ../data/metastore/employee_id_bucketed/*.parquet

In [29]:
spark.table('employee_id_bucketed').show(10)

+-------+----------+-------------------+---+-------+
|     ID| firstname|           lastname|age| salary|
+-------+----------+-------------------+---+-------+
| 417807|  Wojciech|         Wróblewski| 58|3324.65|
|3239618|   Wisława|               Glut| 41|3575.41|
|2298942|  Wojciech|         Malinowski| 28|3762.67|
| 410946|  Wojciech|       Mieczykowski| 49|3814.12|
|1461953| Krzysztof|              Pysla| 60|3827.21|
|1013703|Mieczysław|              Pysla| 18|3856.27|
| 962875|   Wisława|           Kowalski| 19|3893.95|
|2373516|  Wojciech|             Wlotka| 45| 3894.5|
|2479188|Mieczysław|         Wróblewski| 18|3947.45|
| 131993|      Adam|Brzęczyszczykiewicz| 21|3983.21|
+-------+----------+-------------------+---+-------+
only showing top 10 rows


In [30]:
# wypisanie tabeli
spark.catalog.listTables()

[Table(name='employee_id_bucketed', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='lucky_employees', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employee_data_split_1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [31]:
# usunięcie tabeli
spark.sql('DROP TABLE employee_id_bucketed')

DataFrame[]

In [32]:
# wypisanie tabeli
spark.catalog.listTables()

[Table(name='lucky_employees', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employee_data_split_1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [33]:
# jeżeli dane, z którymi pracujemy zawierają stosunkowo niewiele różnorodnych wartości w danych kolumnach
# lub filtrowanie i obliczenia często odbywają się na podgrupach danych to lepsze efekty uzyskamy
# poprzez wykorzystanie możliwości partycjonowania tych danych, które to partycjonowanie
# będzie również odzwierciedlone w fizycznej strukturze plików na dysku twardym w hurtowni danych

# zobaczmy przykład poniżej

df.write.partitionBy("lastname").mode('overwrite').saveAsTable("employees_partitioned_lastname")

In [34]:
# dobrym pomysłem jest też określenie ilości bucketów wynikających z danych w konkretnej kolumnie
# i wykorzystanie do podziału
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.bucketBy.html
buckets = spark.sql("select distinct firstname from EMPLOYEE_DATA").count()
buckets

10

In [35]:
# widok danych podzielonych na partycję z punktu widzenia systemu plików
!ls ../data/metastore/employees_partitioned_lastname

_SUCCESS
lastname=BaraĹ„ski
lastname=BrzÄ™czyszczykiewicz
lastname=Glut
lastname=Kowalski
lastname=Malinowski
lastname=Mieczykowski
lastname=Pysla
lastname=Szczaw
lastname=Wlotka
lastname=WrĂłblewski


In [36]:
df.filter(df.lastname == 'Pysla').groupby('lastname').agg({'salary': 'avg'}).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[lastname#19], functions=[avg(salary#21)])
   +- Exchange hashpartitioning(lastname#19, 200), ENSURE_REQUIREMENTS, [plan_id=764]
      +- HashAggregate(keys=[lastname#19], functions=[partial_avg(salary#21)])
         +- Filter (isnotnull(lastname#19) AND (lastname#19 = Pysla))
            +- FileScan csv [lastname#19,salary#21] Batched: false, DataFilters: [isnotnull(lastname#19), (lastname#19 = Pysla)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Krzysztof/PycharmProjects/pyspark_py3.11/data/employee...., PartitionFilters: [], PushedFilters: [IsNotNull(lastname), EqualTo(lastname,Pysla)], ReadSchema: struct<lastname:string,salary:double>




In [37]:
%%time
df.filter(df.lastname == 'Pysla').groupby('lastname').agg({'salary': 'avg'}).show(10)

+--------+-----------------+
|lastname|      avg(salary)|
+--------+-----------------+
|   Pysla|7849.735708064027|
+--------+-----------------+

CPU times: total: 15.6 ms
Wall time: 26 s


In [38]:
spark.sql("select lastname, avg(salary) from employees_partitioned_lastname where lastname='Pysla' group by lastname").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[lastname#441], functions=[avg(salary#440)])
   +- Exchange hashpartitioning(lastname#441, 200), ENSURE_REQUIREMENTS, [plan_id=831]
      +- HashAggregate(keys=[lastname#441], functions=[partial_avg(salary#440)])
         +- FileScan parquet spark_catalog.default.employees_partitioned_lastname[salary#440,lastname#441] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Krzysztof/PycharmProjects/pyspark_py3.11/data/metastore..., PartitionFilters: [isnotnull(lastname#441), (lastname#441 = Pysla)], PushedFilters: [], ReadSchema: struct<salary:double>




In [39]:
%%time
spark.sql("select lastname, avg(salary) from employees_partitioned_lastname where lastname='Pysla' group by lastname").show(10)

+--------+-----------------+
|lastname|      avg(salary)|
+--------+-----------------+
|   Pysla|7849.735708064312|
+--------+-----------------+

CPU times: total: 0 ns
Wall time: 1.19 s


Jak widać, operacja wykonała się szybciej.

In [7]:
spark.sparkContext.stop()

### Zadania

**Zadanie 1**  
Pamiętacie plik zamówienia.txt?
Plik został umieszczony w folderze z labem w repozytorium.

Wczytaj ten plik za pomocą Sparka do dowolnego typu danych (RDD, Spark DataFrame) i dokonaj transformacji tak aby:
* naprawić problemy z kodowaniem znaków (replace?) w kolumnie Sprzedawca
* poprawić format danych w kolumnie Utarg
* dodać odpowiednie typy danych
* kolumna idZamowienia powinna być traktowana jako klucz (indeks)

**Zadanie 2**  
Po wykonaniu zadania 1, wykorzystaj przykłady z laboratorium i:
* 2.1 wykonaj wiaderkowanie danych i wykonaj dowolne zapytanie agregujące na tych danych vs. dane niepodzielone na wiaderka - porównaj czas
* 2.2 wykonaj partycjonowanie danych i zapisz je w formcie csv (wypróbuj partycjonowanie wg. kraju, nazwiska)
* 2.3 wykonaj zapytanie agregujące z filtrowaniem po kolumnie, której użyłeś/-aś do partycjonowania na danych oryginalnych oraz partycjonowanych i porównaj czas wykonania

**Zadanie 3**  
Z danych wygeneruj 4 różne podzbiory próbek (wiersze wybrane losowo) i dodaj nową kolumnę w każdym z nich, np. w jednym stwórz kolumnę month wyciągając tylko miesiąc z daty, w drugim wartość netto zamówienia (przyjmując, że vat to 23%), w kolejnym zamień nazwisko na wielkie litery, w kolejnym dodaj kolumnę waluta z wartością PLN.

Następnie zapisz każdy z tych zbiorów tak, że:
* zbiór pierwszy to będzie tymczasowa tabela in-memory Sparka
* zbiór drugi to plik(i) parquet
* zbiór trzeci to plik(i) csv
* zbiór czwarty to plik(i) json

Wykonaj zapytanie złączające jak w przykładzie pobierając dane bezpośrednio z plików i wyświetl idZamowienia, Kraj, Sprzedawcę, Datę, Utarg oraz 4 nowo utworzone kolumny.


## Rozwiązania zadań

### Zadanie 1

In [42]:
# wczytanie danych
zam = spark.read.csv('../data/zamowienia.txt', header=True, inferSchema=True, sep=';')

In [43]:
zam.show(10)

+------+----------+---------------+------------+-----------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|      Utarg|
+------+----------+---------------+------------+-----------+
|Polska|  Kowalski|     16.07.2003|       10248|  440,00 z|
|Polska|  Sowiäski|     10.07.2003|       10249|1 863,40 z|
|Niemcy|   Peacock|     12.07.2003|       10250|1 552,60 z|
|Niemcy| Leverling|     15.07.2003|       10251|  654,06 z|
|Niemcy|   Peacock|     11.07.2003|       10252|3 597,90 z|
|Niemcy| Leverling|     16.07.2003|       10253|1 444,80 z|
|Polska|  Kowalski|     23.07.2003|       10254|  556,62 z|
|Polska|     Dudek|     15.07.2003|       10255|2 490,50 z|
|Niemcy| Leverling|     17.07.2003|       10256|  517,80 z|
|Niemcy|   Peacock|     22.07.2003|       10257|1 119,90 z|
+------+----------+---------------+------------+-----------+
only showing top 10 rows


In [44]:
# inna forma zwrócenia tych samych danych
zam.take(10)

[Row(Kraj='Polska', Sprzedawca='Kowalski', Data zamowienia='16.07.2003', idZamowienia=10248, Utarg='440,00 z\x88'),
 Row(Kraj='Polska', Sprzedawca='Sowiäski', Data zamowienia='10.07.2003', idZamowienia=10249, Utarg='1 863,40 z\x88'),
 Row(Kraj='Niemcy', Sprzedawca='Peacock', Data zamowienia='12.07.2003', idZamowienia=10250, Utarg='1 552,60 z\x88'),
 Row(Kraj='Niemcy', Sprzedawca='Leverling', Data zamowienia='15.07.2003', idZamowienia=10251, Utarg='654,06 z\x88'),
 Row(Kraj='Niemcy', Sprzedawca='Peacock', Data zamowienia='11.07.2003', idZamowienia=10252, Utarg='3 597,90 z\x88'),
 Row(Kraj='Niemcy', Sprzedawca='Leverling', Data zamowienia='16.07.2003', idZamowienia=10253, Utarg='1 444,80 z\x88'),
 Row(Kraj='Polska', Sprzedawca='Kowalski', Data zamowienia='23.07.2003', idZamowienia=10254, Utarg='556,62 z\x88'),
 Row(Kraj='Polska', Sprzedawca='Dudek', Data zamowienia='15.07.2003', idZamowienia=10255, Utarg='2 490,50 z\x88'),
 Row(Kraj='Niemcy', Sprzedawca='Leverling', Data zamowienia='17.0

1. naprawić problemy z kodowaniem znaków (replace?) w kolumnie Sprzedawca

In [45]:
# unikalne wartości z kolumny sprzedawca w celu identyfikacji problemów
zam.select(zam.Sprzedawca).distinct().show()

+----------+
|Sprzedawca|
+----------+
|   Peacock|
|      King|
|     Dudek|
|   Davolio|
|    Fuller|
|  Sowiäski|
| Leverling|
|  Kowalski|
|  Callahan|
+----------+



In [46]:
# nie zastało w treści zadania wskazane jaki jest problem, ale polega on na tym, że nazwisko Sowiäski
# powinni brzmieć Sowiński

# wyświetlenie kolumny z poprawnymi wartościami
zam.select(zam.Sprzedawca).replace('Sowiäski', 'Sowiński').show()

+----------+
|Sprzedawca|
+----------+
|  Kowalski|
|  Sowiński|
|   Peacock|
| Leverling|
|   Peacock|
| Leverling|
|  Kowalski|
|     Dudek|
| Leverling|
|   Peacock|
|   Davolio|
|   Peacock|
|   Peacock|
|   Peacock|
|  Callahan|
|     Dudek|
|  Sowiński|
|    Fuller|
| Leverling|
|   Peacock|
+----------+
only showing top 20 rows


In [47]:
# ale trzeba to jeszcze zamienić w miejscu tej kolumny
from pyspark.sql.functions import regexp_replace

zam = zam.withColumn('Sprzedawca', regexp_replace('Sprzedawca', 'Sowiäski', 'Sowiński'))

2. poprawić format danych w kolumnie Utarg

In [48]:
zam.printSchema()

root
 |-- Kraj: string (nullable = true)
 |-- Sprzedawca: string (nullable = true)
 |-- Data zamowienia: string (nullable = true)
 |-- idZamowienia: integer (nullable = true)
 |-- Utarg: string (nullable = true)



In [49]:
zam.select(zam.Utarg).show()

+-----------+
|      Utarg|
+-----------+
|  440,00 z|
|1 863,40 z|
|1 552,60 z|
|  654,06 z|
|3 597,90 z|
|1 444,80 z|
|  556,62 z|
|2 490,50 z|
|  517,80 z|
|1 119,90 z|
|1 614,88 z|
|  100,80 z|
|1 504,65 z|
|  448,00 z|
|  584,00 z|
|1 873,80 z|
|  695,62 z|
|1 176,00 z|
|  346,56 z|
|3 536,60 z|
+-----------+
only showing top 20 rows


In [50]:
# tu można zastosować własną funkcję (lambdę) do obcięcia znaków waluty w kolumnie Utarg
from pyspark.sql import functions as sf

cut_utarg = sf.udf(lambda s: s[:-3])

# zanim faktycznie podstawimy wartości w kolumnie możemy sprawdzić czy jest OK
zam.withColumn("Utarg", cut_utarg("Utarg")).show()

+------+----------+---------------+------------+--------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|   Utarg|
+------+----------+---------------+------------+--------+
|Polska|  Kowalski|     16.07.2003|       10248|  440,00|
|Polska|  Sowiński|     10.07.2003|       10249|1 863,40|
|Niemcy|   Peacock|     12.07.2003|       10250|1 552,60|
|Niemcy| Leverling|     15.07.2003|       10251|  654,06|
|Niemcy|   Peacock|     11.07.2003|       10252|3 597,90|
|Niemcy| Leverling|     16.07.2003|       10253|1 444,80|
|Polska|  Kowalski|     23.07.2003|       10254|  556,62|
|Polska|     Dudek|     15.07.2003|       10255|2 490,50|
|Niemcy| Leverling|     17.07.2003|       10256|  517,80|
|Niemcy|   Peacock|     22.07.2003|       10257|1 119,90|
|Niemcy|   Davolio|     23.07.2003|       10258|1 614,88|
|Niemcy|   Peacock|     25.07.2003|       10259|  100,80|
|Niemcy|   Peacock|     29.07.2003|       10260|1 504,65|
|Niemcy|   Peacock|     30.07.2003|       10261|  448,00|
|Niemcy|  Call

In [51]:
# i zamieniamy wartość w kolumnie
zam = zam.withColumn("Utarg", cut_utarg("Utarg"))
zam.show()

+------+----------+---------------+------------+--------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|   Utarg|
+------+----------+---------------+------------+--------+
|Polska|  Kowalski|     16.07.2003|       10248|  440,00|
|Polska|  Sowiński|     10.07.2003|       10249|1 863,40|
|Niemcy|   Peacock|     12.07.2003|       10250|1 552,60|
|Niemcy| Leverling|     15.07.2003|       10251|  654,06|
|Niemcy|   Peacock|     11.07.2003|       10252|3 597,90|
|Niemcy| Leverling|     16.07.2003|       10253|1 444,80|
|Polska|  Kowalski|     23.07.2003|       10254|  556,62|
|Polska|     Dudek|     15.07.2003|       10255|2 490,50|
|Niemcy| Leverling|     17.07.2003|       10256|  517,80|
|Niemcy|   Peacock|     22.07.2003|       10257|1 119,90|
|Niemcy|   Davolio|     23.07.2003|       10258|1 614,88|
|Niemcy|   Peacock|     25.07.2003|       10259|  100,80|
|Niemcy|   Peacock|     29.07.2003|       10260|1 504,65|
|Niemcy|   Peacock|     30.07.2003|       10261|  448,00|
|Niemcy|  Call

In [ ]:
# teraz próba zamiany typu danych w kolumnie
zam.withColumn("Utarg", sf.col("Utarg").cast("decimal(10,2)")).show()

In [ ]:
# jeżeli separatorem części dziesiętnej są przecinki niestety rzutowanie na typ decimal nie wykona się automatycznie

In [40]:
# z racji tego, że funkcja withColumn zwraca kolumnę dataframe, to możemy łańcuchować wywowałnie funkcji
# zamieniamy przecinek na kropkę oraz pozbywamy się spacji
zam.withColumn('Utarg', regexp_replace('Utarg', ',', '.')).withColumn('Utarg', regexp_replace('Utarg', ' ', '')).show()

+------+----------+---------------+------------+-------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|
+------+----------+---------------+------------+-------+
|Polska|  Kowalski|     16.07.2003|       10248| 440.00|
|Polska|  Sowiński|     10.07.2003|       10249|1863.40|
|Niemcy|   Peacock|     12.07.2003|       10250|1552.60|
|Niemcy| Leverling|     15.07.2003|       10251| 654.06|
|Niemcy|   Peacock|     11.07.2003|       10252|3597.90|
|Niemcy| Leverling|     16.07.2003|       10253|1444.80|
|Polska|  Kowalski|     23.07.2003|       10254| 556.62|
|Polska|     Dudek|     15.07.2003|       10255|2490.50|
|Niemcy| Leverling|     17.07.2003|       10256| 517.80|
|Niemcy|   Peacock|     22.07.2003|       10257|1119.90|
|Niemcy|   Davolio|     23.07.2003|       10258|1614.88|
|Niemcy|   Peacock|     25.07.2003|       10259| 100.80|
|Niemcy|   Peacock|     29.07.2003|       10260|1504.65|
|Niemcy|   Peacock|     30.07.2003|       10261| 448.00|
|Niemcy|  Callahan|     25.07.2

In [52]:
# zapisujemy zmiany
zam = zam.withColumn('Utarg', regexp_replace('Utarg', ',', '.')).withColumn('Utarg', regexp_replace('Utarg', ' ', ''))

In [55]:
# i ponownie próbujemy wykonać rzutowanie
zam = zam.withColumn("Utarg", sf.col("Utarg").cast("decimal(10,2)"))

In [56]:
zam.printSchema()

root
 |-- Kraj: string (nullable = true)
 |-- Sprzedawca: string (nullable = true)
 |-- Data zamowienia: string (nullable = true)
 |-- idZamowienia: integer (nullable = true)
 |-- Utarg: decimal(10,2) (nullable = true)



3. dodać odpowiednie typy danych

In [64]:
# to właściwie zostało wykonane w kroku poprzednim
# datę możemy zostawić jako łańcuch znaków, ale gdybyśmy chcieli jednak zamienić to można tak
# zapiszę do drugiej ramki danych, aby pokazać dwa sposoby wyciągnięcia miesiąca z daty
zam2 = zam.withColumn('Data zamowienia', sf.to_date('Data zamowienia', 'dd.MM.yyyy'))

In [65]:
zam2.show()

+------+----------+---------------+------------+-------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|
+------+----------+---------------+------------+-------+
|Polska|  Kowalski|     2003-07-16|       10248| 440.00|
|Polska|  Sowiński|     2003-07-10|       10249|1863.40|
|Niemcy|   Peacock|     2003-07-12|       10250|1552.60|
|Niemcy| Leverling|     2003-07-15|       10251| 654.06|
|Niemcy|   Peacock|     2003-07-11|       10252|3597.90|
|Niemcy| Leverling|     2003-07-16|       10253|1444.80|
|Polska|  Kowalski|     2003-07-23|       10254| 556.62|
|Polska|     Dudek|     2003-07-15|       10255|2490.50|
|Niemcy| Leverling|     2003-07-17|       10256| 517.80|
|Niemcy|   Peacock|     2003-07-22|       10257|1119.90|
|Niemcy|   Davolio|     2003-07-23|       10258|1614.88|
|Niemcy|   Peacock|     2003-07-25|       10259| 100.80|
|Niemcy|   Peacock|     2003-07-29|       10260|1504.65|
|Niemcy|   Peacock|     2003-07-30|       10261| 448.00|
|Niemcy|  Callahan|     2003-07

4. kolumna idZamowienia powinna być traktowana jako klucz (indeks)

In [71]:
# zadanie powinno polegać na wywołaniu poniższej funkcji i ustawieniu tej kolumny jako indeksu, jednak nie działa poprawnie
# mimo, że w dokumentacji API PySpark 4.0.1 występuje wraz z przykładem
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.set_index.html
# zam.set_index('idZamowienia')

# EDIT: dopiero po czasie zorientowałem się, że dotyczy do Pandas API dla Spark, a to jednak nie to samo ;-)

# ale można spróbować tak
zam.index = zam.idZamowienia

In [73]:
zam.show()

+------+----------+---------------+------------+-------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|
+------+----------+---------------+------------+-------+
|Polska|  Kowalski|     16.07.2003|       10248| 440.00|
|Polska|  Sowiński|     10.07.2003|       10249|1863.40|
|Niemcy|   Peacock|     12.07.2003|       10250|1552.60|
|Niemcy| Leverling|     15.07.2003|       10251| 654.06|
|Niemcy|   Peacock|     11.07.2003|       10252|3597.90|
|Niemcy| Leverling|     16.07.2003|       10253|1444.80|
|Polska|  Kowalski|     23.07.2003|       10254| 556.62|
|Polska|     Dudek|     15.07.2003|       10255|2490.50|
|Niemcy| Leverling|     17.07.2003|       10256| 517.80|
|Niemcy|   Peacock|     22.07.2003|       10257|1119.90|
|Niemcy|   Davolio|     23.07.2003|       10258|1614.88|
|Niemcy|   Peacock|     25.07.2003|       10259| 100.80|
|Niemcy|   Peacock|     29.07.2003|       10260|1504.65|
|Niemcy|   Peacock|     30.07.2003|       10261| 448.00|
|Niemcy|  Callahan|     25.07.2

In [74]:
zam.index

Column<'idZamowienia'>

### Zadanie 2

In [78]:
# wiaderkowanie po kraju
zam.write.bucketBy(2, 'Kraj').mode('overwrite').sortBy('Sprzedawca').saveAsTable('zamowienia_kraj_bucketed')

In [80]:
# folder istnieje
!dir ..\data\metastore

 Volume in drive C is Windows_OS
 Volume Serial Number is 06D6-CDEC

 Directory of C:\Users\Krzysztof\PycharmProjects\pyspark_py3.11\data\metastore

12.10.2025  13:26    <DIR>          .
12.10.2025  13:26    <DIR>          ..
11.10.2025  15:08    <DIR>          employees_partitioned_lastname
11.10.2025  14:54    <DIR>          lucky_employees
12.10.2025  13:26    <DIR>          zamowienia_kraj_bucketed
               0 File(s)              0 bytes
               5 Dir(s)  23˙394˙086˙912 bytes free


In [85]:
%%time
# zapytanie agregujące na oryginalnej dataframe
zam.filter(zam.Kraj == 'Polska').groupby('Sprzedawca').agg({'Utarg': 'avg'}).show()

+----------+-----------+
|Sprzedawca| avg(Utarg)|
+----------+-----------+
|  Sowiński|1115.809692|
|      King|1745.716269|
|     Dudek|1830.440000|
|  Kowalski|1637.910714|
+----------+-----------+

CPU times: total: 0 ns
Wall time: 1.16 s


In [89]:
%%time
# zapytanie do danych zapisanych w hurtowni Hive (to te w folderze ../data/metastore)
spark.sql("select Sprzedawca, avg(Utarg) from zamowienia_kraj_bucketed where Kraj='Polska' group by Sprzedawca").show()

+----------+-----------+
|Sprzedawca| avg(Utarg)|
+----------+-----------+
|  Sowiński|1115.809692|
|      King|1745.716269|
|     Dudek|1830.440000|
|  Kowalski|1637.910714|
+----------+-----------+

CPU times: total: 0 ns
Wall time: 235 ms


In [91]:
# 2.2 partycjonowanie i zapis w formacie csv
zam.write.partitionBy("Kraj").mode('overwrite').saveAsTable("zamowienia_partition_by_kraj", format='csv')
zam.write.partitionBy("Sprzedawca").mode('overwrite').saveAsTable("zamowienia_partition_by_sprzedawca", format='csv')

In [92]:
%%time
zam.filter(zam.Kraj == 'Niemcy').groupby('Sprzedawca').agg({'Utarg': 'avg'}).show()

+----------+-----------+
|Sprzedawca| avg(Utarg)|
+----------+-----------+
|   Peacock|1495.123709|
|   Davolio|1559.829829|
|    Fuller|1766.345435|
| Leverling|1609.570160|
|  Callahan|1242.754242|
+----------+-----------+

CPU times: total: 0 ns
Wall time: 1.29 s


In [95]:
%%time
spark.sql("select Sprzedawca, avg(Utarg) from zamowienia_partition_by_kraj where Kraj='Niemcy' group by Sprzedawca").show()

+----------+-----------+
|Sprzedawca| avg(Utarg)|
+----------+-----------+
|   Peacock|1495.123709|
|   Davolio|1559.829829|
|    Fuller|1766.345435|
| Leverling|1609.570160|
|  Callahan|1242.754242|
+----------+-----------+

CPU times: total: 0 ns
Wall time: 203 ms


In [94]:
%%time
spark.sql("select Sprzedawca, avg(Utarg) from zamowienia_partition_by_sprzedawca where Kraj='Niemcy' group by Sprzedawca").show()

+----------+-----------+
|Sprzedawca| avg(Utarg)|
+----------+-----------+
|   Peacock|1495.123709|
|   Davolio|1559.829829|
|    Fuller|1766.345435|
| Leverling|1609.570160|
|  Callahan|1242.754242|
+----------+-----------+

CPU times: total: 0 ns
Wall time: 240 ms


### Zadanie 3

In [132]:
# podział danych - wykorzystamy sample
splits = [zam.sample(fraction=0.25) for n in range(4)]

In [133]:
for num, split in enumerate(splits, start=1):
    print(f"Liczebność splitu {num}: {split.count()}")

Liczebność splitu 1: 189
Liczebność splitu 2: 221
Liczebność splitu 3: 214
Liczebność splitu 4: 208


In [134]:
# dla przypomnienia dane
splits[0].show(10)

+------+----------+---------------+------------+-------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|
+------+----------+---------------+------------+-------+
|Niemcy| Leverling|     16.07.2003|       10253|1444.80|
|Niemcy| Leverling|     17.07.2003|       10256| 517.80|
|Niemcy| Leverling|     31.07.2003|       10266| 346.56|
|Niemcy|   Davolio|     02.08.2003|       10270|1376.00|
|Niemcy|   Davolio|     09.08.2003|       10275| 291.84|
|Niemcy|  Callahan|     14.08.2003|       10276| 420.00|
|Niemcy|  Callahan|     16.08.2003|       10279| 351.00|
|Niemcy|   Peacock|     27.08.2003|       10284|1170.37|
|Niemcy|   Peacock|     05.09.2003|       10294|1887.60|
|Niemcy|   Peacock|     13.09.2003|       10299| 349.50|
+------+----------+---------------+------------+-------+
only showing top 10 rows


In [135]:
# dodanie nowych kolumn do splitów
# split 1 - dodatkowa kolumna w postaci wartości miesiąca wyciągniętej z kolumny data
# sposób 1 - jeżeli kolumna pozostała typem string, a widzimy, że wartości są z poprzedzającym zerem to pobieramy 2 znaki począwszy od 4 znaku
splits[0] = splits[0].withColumn('month', sf.substring('Data zamowienia',4,2))

In [136]:
splits[0].show()

+------+----------+---------------+------------+-------+-----+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|month|
+------+----------+---------------+------------+-------+-----+
|Niemcy| Leverling|     16.07.2003|       10253|1444.80|   07|
|Niemcy| Leverling|     17.07.2003|       10256| 517.80|   07|
|Niemcy| Leverling|     31.07.2003|       10266| 346.56|   07|
|Niemcy|   Davolio|     02.08.2003|       10270|1376.00|   08|
|Niemcy|   Davolio|     09.08.2003|       10275| 291.84|   08|
|Niemcy|  Callahan|     14.08.2003|       10276| 420.00|   08|
|Niemcy|  Callahan|     16.08.2003|       10279| 351.00|   08|
|Niemcy|   Peacock|     27.08.2003|       10284|1170.37|   08|
|Niemcy|   Peacock|     05.09.2003|       10294|1887.60|   09|
|Niemcy|   Peacock|     13.09.2003|       10299| 349.50|   09|
|Niemcy|    Fuller|     18.09.2003|       10300| 608.00|   09|
|Niemcy|   Davolio|     23.09.2003|       10306| 498.50|   09|
|Polska|      King|     24.09.2003|       10308|  88.80

In [137]:
# sposób 2 - jeżeli kolumna została zamieniona na typ date
# jest to oczywiście dużo lepszy pomysł, zwłaszcza przy filtrowaniu danych na podstawie takiej kolumny
zam2.withColumn('month', sf.month('Data zamowienia')).show()

+------+----------+---------------+------------+-------+-----+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|month|
+------+----------+---------------+------------+-------+-----+
|Polska|  Kowalski|     2003-07-16|       10248| 440.00|    7|
|Polska|  Sowiński|     2003-07-10|       10249|1863.40|    7|
|Niemcy|   Peacock|     2003-07-12|       10250|1552.60|    7|
|Niemcy| Leverling|     2003-07-15|       10251| 654.06|    7|
|Niemcy|   Peacock|     2003-07-11|       10252|3597.90|    7|
|Niemcy| Leverling|     2003-07-16|       10253|1444.80|    7|
|Polska|  Kowalski|     2003-07-23|       10254| 556.62|    7|
|Polska|     Dudek|     2003-07-15|       10255|2490.50|    7|
|Niemcy| Leverling|     2003-07-17|       10256| 517.80|    7|
|Niemcy|   Peacock|     2003-07-22|       10257|1119.90|    7|
|Niemcy|   Davolio|     2003-07-23|       10258|1614.88|    7|
|Niemcy|   Peacock|     2003-07-25|       10259| 100.80|    7|
|Niemcy|   Peacock|     2003-07-29|       10260|1504.65

In [138]:
# split 2 - zmiana wartości Utarg na netto
splits[1] = splits[1].withColumn('Utarg netto', sf.round(splits[1].Utarg / 1.23, 2))

In [139]:
# split 3 - nazwisko na wielkie litery
splits[2] = splits[2].withColumn('Sprzedawca upper', sf.upper(splits[2].Sprzedawca))

In [140]:
# split 4 - nowa kolumna z wartością PLN
splits[3] = splits[3].withColumn('Waluta', sf.lit('PLN'))

In [141]:
# tabela w pamięci
splits[0].createOrReplaceTempView("zamowienia_split_1")
# parquet
splits[1].write.mode('overwrite').saveAsTable("zamowienia_split_2", format='parquet')
# csv
splits[2].write.mode('overwrite').saveAsTable("zamowienia_split_3", format='csv')
# json
splits[3].write.mode('overwrite').saveAsTable("zamowienia_split_4", format='json')

In [143]:
# zapytanie złączające - złączenie będziemy wykonywać po kolumnie idZamowienia - część na pewno się pokryła w procesie losowania
query = """
SELECT  zam1.*,
        `zam2`.`Utarg netto`,
        `zam3`.`Sprzedawca upper`,
        zam4.Waluta
FROM zamowienia_split_1 zam1 
JOIN zamowienia_split_2 zam2 on zam1.idZamowienia=zam2.idZamowienia
JOIN zamowienia_split_3 zam3 on zam1.idZamowienia=zam3.idZamowienia
JOIN zamowienia_split_4 zam4 on zam1.idZamowienia=zam4.idZamowienia
"""
spark.sql(query).show(10)

+------+----------+---------------+------------+-------+-----+-----------+----------------+------+
|  Kraj|Sprzedawca|Data zamowienia|idZamowienia|  Utarg|month|Utarg netto|Sprzedawca upper|Waluta|
+------+----------+---------------+------------+-------+-----+-----------+----------------+------+
|Niemcy|    Fuller|     18.09.2003|       10300| 608.00|   09|     494.31|          FULLER|   PLN|
|Niemcy|   Peacock|     02.09.2004|       10645|1535.00|   09|    1247.97|         PEACOCK|   PLN|
|Polska|  Kowalski|     13.02.2005|       10870| 160.00|   02|     130.08|        KOWALSKI|   PLN|
|Niemcy| Leverling|     18.03.2005|       10936| 456.00|   03|     370.73|       LEVERLING|   PLN|
|Niemcy|   Davolio|     06.04.2005|       10995|1196.00|   04|     972.36|         DAVOLIO|   PLN|
+------+----------+---------------+------------+-------+-----+-----------+----------------+------+

